# TSP-2000 日志解析与对比分析

本 notebook 分析 4 个方法在 TSP-2000 数据集上的表现：
- **ELG**
- **LEHD**
- **OMNI**
- **ICAM**

## 分析重点：
1. 各方法的总体性能（平均 Augmented Score）
2. 每个实例的最优方法及其领先数量
3. 各方法领先的具体实例列表

In [ ]:
# 导入必要的库
from __future__ import annotations
from pathlib import Path
import re
from collections import defaultdict
import numpy as np
import pandas as pd

# 设置
REPO_ROOT = Path('/mnt/d/Study/3/neural-solver-selection')
LOGS_DIR = REPO_ROOT / 'results/benchmarks/tsp_datasets/logs'

# 4 个方法的 TSP-2000 日志文件
METHODS = ['elg', 'lehd', 'omni', 'icam']
LOG_FILES = {
    'elg': LOGS_DIR / 'elg' / 'test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log',
    'lehd': LOGS_DIR / 'lehd' / 'test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log',
    'omni': LOGS_DIR / 'omni' / 'test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log',
    'icam': LOGS_DIR / 'icam' / 'test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log',
}

print(f"日志目录: {LOGS_DIR}")
for method, path in LOG_FILES.items():
    exists = "✓" if path.exists() else "✗"
    print(f"  {exists} {method}: {path.name}")

## 1. 日志解析函数

In [ ]:
def parse_tsp2000_log(log_path: Path, method: str) -> dict[str, dict]:
    """
    解析 TSP-2000 日志文件，提取每个实例的 Score 和 Augmented Score
    
    返回格式: {instance_id: {'score': float, 'aug_score': float}}
    """
    results = {}
    
    # 日志行格式:
    # [2026-01-20 06:16:01,065]... Eval ==> Problem[tsp], Size[2000], Completed   1/128(0.78%), Score: 37.6226, Augmented Score: 37.2747
    pattern = r"Completed\s+(\d+)/128.*Score:\s+([\d.]+),\s+Augmented Score:\s+([\d.]+)"
    
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            match = re.search(pattern, line)
            if match:
                instance_id = int(match.group(1))
                score = float(match.group(2))
                aug_score = float(match.group(3))
                results[instance_id] = {
                    'score': score,
                    'aug_score': aug_score
                }
    
    print(f"{method}: 解析了 {len(results)} 个实例")
    return results

# 解析所有方法
all_results = {}
for method in METHODS:
    all_results[method] = parse_tsp2000_log(LOG_FILES[method], method)

print("\n所有方法解析完成！")
print(f"总实例数: {len(all_results['elg'])}")

## 2. 构建对比数据表

In [ ]:
# 构建对比表
def build_comparison_table(results: dict[str, dict]) -> pd.DataFrame:
    """
    构建所有方法的对比表
    """
    rows = []
    instance_ids = sorted(results['elg'].keys())
    
    for iid in instance_ids:
        row = {'instance_id': iid}
        for method in METHODS:
            if iid in results[method]:
                row[f'{method}_score'] = results[method][iid]['score']
                row[f'{method}_aug'] = results[method][iid]['aug_score']
            else:
                row[f'{method}_score'] = None
                row[f'{method}_aug'] = None
        rows.append(row)
    
    return pd.DataFrame(rows)

df = build_comparison_table(all_results)
print(f"对比表形状: {df.shape}")
df.head(10)

## 3. 各方法总体统计

In [ ]:
# 计算各方法的平均性能
summary_stats = []

for method in METHODS:
    aug_scores = [all_results[method][iid]['aug_score'] 
                    for iid in all_results[method].keys()]
    scores = [all_results[method][iid]['score'] 
              for iid in all_results[method].keys()]
    
    summary_stats.append({
        'Method': method.upper(),
        'Num_Instances': len(aug_scores),
        'Avg_Score': np.mean(scores),
        'Std_Score': np.std(scores),
        'Avg_Aug_Score': np.mean(aug_scores),
        'Std_Aug_Score': np.std(aug_scores),
        'Min_Aug_Score': np.min(aug_scores),
        'Max_Aug_Score': np.max(aug_scores),
    })

summary_df = pd.DataFrame(summary_stats)
summary_df = summary_df.sort_values('Avg_Aug_Score')

print("=== 各方法总体统计（按 Avg_Aug_Score 排序）===")
print(summary_df.to_string(index=False))

# 保存到变量供后续使用
best_method = summary_df.iloc[0]['Method']
best_avg_score = summary_df.iloc[0]['Avg_Aug_Score']
print(f"\n最优方法: {best_method}, 平均 Aug Score: {best_avg_score:.4f}")

## 4. 每个实例的最优方法

In [ ]:
# 找出每个实例的最优方法（基于 Augmented Score）
def find_best_method_per_instance(df: pd.DataFrame) -> pd.DataFrame:
    """
    为每个实例找出最优方法及其领先情况
    """
    results = []
    
    for _, row in df.iterrows():
        iid = row['instance_id']
        
        # 收集所有方法的 aug_score
        scores = {}
        for method in METHODS:
            aug = row[f'{method}_aug']
            if pd.notna(aug):
                scores[method] = aug
        
        if not scores:
            continue
        
        # 找最优和次优
        sorted_methods = sorted(scores.items(), key=lambda x: x[1])
        best_method, best_score = sorted_methods[0]
        
        # 计算领先幅度
        if len(sorted_methods) > 1:
            second_score = sorted_methods[1][1]
            lead = second_score - best_score
            lead_pct = (lead / second_score) * 100
        else:
            lead = 0
            lead_pct = 0
        
        # 检查是否有并列
        ties = [m for m, s in sorted_methods if abs(s - best_score) < 1e-6]
        
        results.append({
            'instance_id': iid,
            'best_method': best_method.upper(),
            'best_aug_score': best_score,
            'lead_over_second': lead,
            'lead_percentage': lead_pct,
            'is_tie': len(ties) > 1,
            'num_methods': len(scores),
        })
    
    return pd.DataFrame(results)

best_df = find_best_method_per_instance(df)

print("每个实例的最优方法（前20个）:")
print(best_df.head(20).to_string(index=False))

## 5. 各方法领先统计

In [ ]:
# 统计各方法领先的实例数量
win_counts = best_df['best_method'].value_counts().reset_index()
win_counts.columns = ['Method', 'Win_Count']
win_counts['Win_Percentage'] = (win_counts['Win_Count'] / len(best_df) * 100).round(2)

print("=== 各方法领先实例数量统计 ===")
print(win_counts.to_string(index=False))

# 可视化
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.bar(win_counts['Method'], win_counts['Win_Count'], color='steelblue')
plt.xlabel('Method')
plt.ylabel('Number of Wins')
plt.title('TSP-2000: Number of Instances Where Each Method Wins')
plt.grid(axis='y', alpha=0.3)
for i, v in enumerate(win_counts['Win_Count']):
    plt.text(i, v + 0.5, str(v), ha='center', fontweight='bold')
plt.show()

## 6. 各方法领先的实例列表

In [ ]:
# 为每个方法列出其领先的实例
for target_method in METHODS:
    method_upper = target_method.upper()
    
    # 筛选该方法领先的实例
    method_wins = best_df[best_df['best_method'] == method_upper].copy()
    method_wins = method_wins.sort_values('lead_percentage', ascending=False)
    
    print(f"\n{'='*60}")
    print(f"{method_upper} 领先的实例 (共 {len(method_wins)} 个)")
    print(f"{'='*60}")
    
    if len(method_wins) == 0:
        print("(无)")
        continue
    
    # 显示前20个
    display_cols = ['instance_id', 'best_aug_score', 'lead_over_second', 'lead_percentage']
    print(method_wins[display_cols].head(20).to_string(index=False))
    
    # 统计信息
    avg_lead = method_wins['lead_percentage'].mean()
    max_lead = method_wins['lead_percentage'].max()
    print(f"\n平均领先幅度: {avg_lead:.2f}%")
    print(f"最大领先幅度: {max_lead:.2f}%")

## 7. 并列最优的实例分析

In [ ]:
# 分析并列情况
tie_instances = best_df[best_df['is_tie'] == True]

if len(tie_instances) > 0:
    print(f"\n=== 并列最优实例分析 ===")
    print(f"共有 {len(tie_instances)} 个实例出现并列最优")
    print(f"\n前10个并列实例:")
    print(tie_instances[['instance_id', 'best_aug_score']].head(10).to_string(index=False))
else:
    print("\n没有并列最优的实例")

## 8. 实例难度分析（各方法表现最差的实例）

In [ ]:
# 找出每个方法表现最差的实例（最高 aug_score）
for method in METHODS:
    method_upper = method.upper()
    
    # 筛选该方法的所有实例
    method_data = df[['instance_id', f'{method}_aug']].dropna()
    method_data = method_data.rename(columns={f'{method}_aug': 'aug_score'})
    
    # 按 aug_score 排序，找出最差的10个
    worst_10 = method_data.nlargest(10, 'aug_score')
    
    print(f"\n{'='*50}")
    print(f"{method_upper} 表现最差的10个实例")
    print(f"{'='*50}")
    print(worst_10.to_string(index=False))

## 9. 方法间两两对比

In [ ]:
# 两两方法对比
def pairwise_comparison(df: pd.DataFrame, method1: str, method2: str) -> dict:
    """
    对比两个方法，返回统计结果
    """
    # 筛选两个方法都有结果的实例
    valid_df = df[[f'{method1}_aug', f'{method2}_aug']].dropna()
    
    m1_wins = (valid_df[f'{method1}_aug'] < valid_df[f'{method2}_aug']).sum()
    m2_wins = (valid_df[f'{method2}_aug'] < valid_df[f'{method1}_aug']).sum()
    ties = (valid_df[f'{method1}_aug'] == valid_df[f'{method2}_aug']).sum()
    
    # 计算平均差异
    avg_diff = (valid_df[f'{method1}_aug'] - valid_df[f'{method2}_aug']).mean()
    
    return {
        'm1': method1.upper(),
        'm2': method2.upper(),
        'm1_wins': m1_wins,
        'm2_wins': m2_wins,
        'ties': ties,
        'total': len(valid_df),
        'm1_better_pct': m1_wins / len(valid_df) * 100,
        'avg_diff': avg_diff,
    }

print("\n=== 两两方法对比 ===\n")

# 生成所有两两对比
comparisons = []
for i, m1 in enumerate(METHODS):
    for m2 in METHODS[i+1:]:
        comp = pairwise_comparison(df, m1, m2)
        comparisons.append(comp)
        
        print(f"{comp['m1']} vs {comp['m2']}:")
        print(f"  {comp['m1']} 胜: {comp['m1_wins']} ({comp['m1_better_pct']:.1f}%)")
        print(f"  {comp['m2']} 胜: {comp['m2_wins']} ({100-comp['m1_better_pct']-comp['ties']/comp['total']*100:.1f}%)")
        print(f"  并列: {comp['ties']}")
        print(f"  平均差异 ({comp['m1']} - {comp['m2']}): {comp['avg_diff']:.4f}")
        print()

## 10. 总结表格导出

In [ ]:
# 导出关键统计表格
export_data = {
    'summary_stats': summary_df,
    'win_counts': win_counts,
    'best_per_instance': best_df,
    'full_comparison': df,
}

# 保存为 CSV
output_dir = REPO_ROOT / 'results/benchmarks/tsp_datasets/analysis'
output_dir.mkdir(parents=True, exist_ok=True)

for name, data in export_data.items():
    output_path = output_dir / f'tsp2000_{name}.csv'
    data.to_csv(output_path, index=False)
    print(f"已保存: {output_path}")

print(f"\n所有分析结果已保存到: {output_dir}")